<a href="https://colab.research.google.com/github/VGoma23/data-520-asean/blob/main/copy_decision_tree_make_rslearn_great_again.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Acknowledgements

this is all based on (copied from) Mikey's existing code in the Github repo for DATA520 ASEAN project.

#Imports

In [1]:
pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 52.5 MB/s eta 0:00:00


In [2]:
import glob
from pathlib import Path
import json
import rasterio
from rasterio.crs import CRS
from rasterio import warp

In [3]:
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='iuu-fishing-detections-asean')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

mikey = '/content/drive/MyDrive/Data_520_Project/data/mikey/mikey'
austin = '/content/drive/MyDrive/Data_520_Project/data/austin/austin'

Mounted at /content/drive


#Code

constants:

In [5]:
bands = ['B12', 'B11', 'B8A', 'B8', 'B7', 'B6', 'B5', 'B4', 'B3', 'B2'] # 5 6 7 8A 11 12
vis_bands = ['B4', 'B3', 'B2']
label = 'vessel'
test_scale = 10

In [14]:
def fetch_image_gee(image_path, test=False):
  metadata = json.load( open(str(image_path / "metadata.json")))
  image_id = open(str(image_path / "image_name_from_siv.txt")).read()[:-5]
  time_start = ee.Date(metadata['time_range'][0])
  time_end = ee.Date(metadata['time_range'][1])

  time_start = time_start.advance(-10, "minute")
  time_end = time_end.advance(10, "minute")

  rectangle_bounds = ee.Geometry.Rectangle(
    [metadata['bounds'][0], metadata['bounds'][1],
      metadata['bounds'][2], metadata['bounds'][3]],
    ee.Projection(metadata['projection']['crs'],
      [10, 0, 0, 0, -10, 0])
    , True, False
    )

  if test:
    image = (
      ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
      .filterDate(time_start, time_end)
      .filter(ee.Filter.stringContains('PRODUCT_ID', image_id))
      .select(bands, bands)
      .first()
      .clip(rectangle_bounds)
    )
  else:
    image = (
      ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
      .filterDate(time_start, time_end)
      .filter(ee.Filter.stringContains('PRODUCT_ID', image_id))
      .select(bands, bands)
    )

  return image, rectangle_bounds


def set_label_0(feature):
  return feature.set(label, 0)


def create_train_with_features(train_path):

  skipped = 0
  image_collection = []
  features_collection = ee.FeatureCollection([])

  for image_path in Path(f"{train_path}").glob("*"):
    image, rectangle_bounds = fetch_image_gee(image_path)

    if (image.size().getInfo() == 0):
      skipped+=1
      continue
    image = image.first().clip(rectangle_bounds)

    features = json.load(open(str(image_path / "layers/label/data.geojson")))['features']
    features_collection_unmapped = []
    for feature in features:
      features_collection_unmapped.append(ee.Feature(ee.Geometry.Point(
      feature['geometry']['coordinates'],
      rectangle_bounds.projection()),
        {label: 1}
      ))
    #if len(features) > 20:
    #  print(f"num features: {len(features)}, filename: {image_path}")
    if len(features) == 0:
      non_vessel_points_unlabeled = image.select(bands).sample(
        region= image.geometry(),
          scale = test_scale,
          numPixels = 20,
          geometries= False,  # Set this to False to ignore geometries
      )
    else:
      non_vessel_points_unlabeled = ee.FeatureCollection([])

    non_vessel_points = non_vessel_points_unlabeled.map(set_label_0)
    # sample x amount of points, add them to features_collection with vessel = 0
    features_collection_unmapped_fc = ee.FeatureCollection(features_collection_unmapped)
    features_collection = features_collection.merge(image.select(bands).sampleRegions(
      collection=features_collection_unmapped_fc, properties=[label], scale=test_scale
    ))

    features_collection = features_collection.merge(non_vessel_points)

    image_collection.append(image)

  master_image = ee.ImageCollection(image_collection)
# master_image
# features_collection
  print(f"skipped: {skipped}")

  return features_collection, master_image


features_collection, master_image = create_train_with_features(mikey)

skipped: 9


In [19]:
def create_GEE_map():
  Map = geemap.Map()
  vis_params = {"min": 0, "max": 3000, "bands": vis_bands}
  Map.addLayer(master_image, vis_params, "combined images example")
  return Map, vis_params

# Map.addLayer(training_features)
Map, vis_params = create_GEE_map()

In [16]:
test_img_folder = Path(f"{austin}") / "1308303_2702491_95728"
test_image, test_rectangle_bounds = fetch_image_gee(test_img_folder, test=True)
test_image

In [17]:
# Train a classifier with default parameters.
# the below classifier can be swapped out pretty easily below
# Note: currently testing smileRandomForest(40, bagFraction=0.35)

# this will inevitably become more complicated later i feel.
def train_model(features_collection):
  return ee.Classifier.smileRandomForest(1).train(features_collection, label, bands)


trained = train_model(features_collection)

# Classify the image with the same bands used for training.
classified = test_image.classify(trained)
classified

classified_reduced = classified.focalMedian(25, 'circle', 'meters',3)

In [22]:
def plot_test_classified(Map, vis_params, test_image, test_rectangle_bounds):

  Map.addLayer(test_image, vis_params, "test classified image")

  Map.add_layer(
      classified,
      {'min': 0, 'max': 1, 'palette': ['blue', 'red']},
      'classification',
  )

  Map.add_layer(
      classified_reduced,
      {'min': 0, 'max': 1, 'palette': ['blue', 'red']},
      'classification_reduced',
  )

  Map.centerObject(test_rectangle_bounds.centroid(), 13)
  return Map

plot_test_classified(Map, vis_params, test_image, test_rectangle_bounds)

Map(bottom=1351849.0, center=[-46.057632981291846, -67.65523212091558], controls=(WidgetControl(options=['posi…